In [0]:
from pyspark.sql.functions import col, coalesce, lit, row_number, concat, substring, expr, current_timestamp
from pyspark.sql.window import Window

# 1. Read from Silver and Gold Geography
df_silver = spark.read.table("hdfc_ergo.hdfc_silver.dim_members")
df_geo = spark.read.table("hdfc_ergo.hdfc_gold.dim_geography")

# 2. Generate Surrogate Key (member_sk)
windowSpec = Window.partitionBy("member_id").orderBy("member_id")
df = df_silver.withColumn("member_sk", row_number().over(windowSpec))

# 3. Lookup geography_sk (Orphan Handling -1)
df = df.join(df_geo.select("state", "city", "geography_sk"), on=["state", "city"], how="left")
df = df.withColumn("geography_sk", coalesce(col("geography_sk").cast("int"), lit(-1)))

# 4. PII Masking (With structural existence safeguards)
aadhar_col = "aadhar_clean" if "aadhar_clean" in df.columns else "aadhar_number"
df = df.withColumn("aadhar_masked", concat(lit("XXXX-XXXX-"), expr(f"right({aadhar_col}, 4)")))

# Safe PAN Handling: Checks if 'pan' exists in the source schema dataframe
if "pan" in df.columns:
    df = df.withColumn("pan_masked", concat(substring(col("pan"), 1, 3), lit("XXXXXXX")))
else:
    df = df.withColumn("pan_masked", lit("NOT PROVIDED"))

name_col = "full_name" if "full_name" in df.columns else "name"
df = df.withColumn("name_masked", concat(substring(col(name_col), 1, 1), lit("***")))

# 5. SCD Type 2 (Initial load defaults wrapped safely in parentheses)
df = (df.withColumn("_valid_from", current_timestamp())
        .withColumn("_valid_to", lit(None).cast("timestamp"))
        .withColumn("_is_current", lit(True)))

# 6. Column Pruning (Select ONLY Gold business columns)
df_gold = df.select(
    "member_sk",
    "member_id",
    "name_masked",
    "aadhar_masked",
    "pan_masked",
    "geography_sk",
    "policy_number",
    "sum_insured",
    "chronic_conditions",
    "language_preference",
    "risk_score",
    "_valid_from",
    "_valid_to",
    "_is_current"
)

# 7. Write to Gold Schema
df_gold.write.mode("overwrite").saveAsTable("hdfc_ergo.hdfc_gold.dim_members")

print("✅ Gold dim_members created successfully!")
display(spark.table("hdfc_ergo.hdfc_gold.dim_members").limit(5))